In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- CONFIG ----
FILEPATH = "./buoy/station44086_onshore_low.txt"  # change if needed
LOCAL_TZ = "America/Los_Angeles"

# NOAA/NDBC-style columns from your header:
COLS = [
    "YY","MM","DD","hh","mm",
    "WDIR","WSPD","GST",
    "WVHT","DPD","APD","MWD",
    "PRES","ATMP","WTMP","DEWP","VIS","TIDE"
]

In [ ]:
def load_buoy_txt(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        comment="#",
        delim_whitespace=True,
        names=COLS,
        engine="python"
    )

    # Convert timestamp to timezone-aware datetime.
    # NDBC historical text is typically UTC; we treat it as UTC then convert to local.
    dt = pd.to_datetime(
        df[["YY","MM","DD","hh","mm"]].rename(columns={"YY":"year","MM":"month","DD":"day","hh":"hour","mm":"minute"}),
        errors="coerce"
    )
    df["time_utc"] = dt.dt.tz_localize("UTC")
    df["time_local"] = df["time_utc"].dt.tz_convert(LOCAL_TZ)

    # Replace common NDBC missing sentinels with NaN:
    # 999, 99, 9999, 99.00 etc show up depending on variable.
    missing_sentinels = {99, 99.0, 999, 999.0, 9999, 9999.0, 99.00}
    for c in COLS:
        if c in ["YY","MM","DD","hh","mm"]:
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c].isin(missing_sentinels), c] = np.nan

    return df.sort_values("time_local").reset_index(drop=True)

def plot_buoy_timeseries(
    df: pd.DataFrame,
    date_local: str,              # "YYYY-MM-DD" in local time
    window_start_local: str=None, # "YYYY-MM-DD HH:MM" local
    window_end_local: str=None    # "YYYY-MM-DD HH:MM" local
):
    # Filter to the local calendar day
    day = pd.Timestamp(date_local).tz_localize(LOCAL_TZ)
    day_end = day + pd.Timedelta(days=1)

    sub = df[(df["time_local"] >= day) & (df["time_local"] < day_end)].copy()
    if sub.empty:
        raise ValueError(f"No rows found for local date {date_local} in {LOCAL_TZ}")

    # Optional shaded window
    w0 = pd.Timestamp(window_start_local).tz_localize(LOCAL_TZ) if window_start_local else None
    w1 = pd.Timestamp(window_end_local).tz_localize(LOCAL_TZ) if window_end_local else None

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

    # Panel A: WVHT
    axes[0].plot(sub["time_local"], sub["WVHT"])
    axes[0].set_ylabel("WVHT (m)")
    axes[0].set_title(f"Buoy forcing (local): {date_local}")

    # Panel B: MWD
    axes[1].plot(sub["time_local"], sub["MWD"])
    axes[1].set_ylabel("MWD (deg)")

    # Panel C: DPD (use APD if you prefer)
    axes[2].plot(sub["time_local"], sub["DPD"])
    axes[2].set_ylabel("DPD (s)")
    axes[2].set_xlabel(f"Time ({LOCAL_TZ})")

    if w0 and w1:
        for ax in axes:
            ax.axvspan(w0, w1, alpha=0.2)
        axes[0].text(
            w0, axes[0].get_ylim()[1],
            "  6-hr window",
            va="top"
        )

    plt.tight_layout()
    plt.show()

# ---- RUN ----
df = load_buoy_txt(FILEPATH)


In [ ]:
# Example: plot a day (edit to your target day)
# If you want to shade your 6-hour window, set window_start_local/window_end_local.
plot_buoy_timeseries(
    df,
    date_local="2024-08-27",
    window_start_local="2024-08-27 10:00",
    window_end_local="2024-08-27 16:00",
)